# 📊 EDA — Eventos culturales de Euskadi (Kulturklik)

Análisis Exploratorio de Datos sobre **toda la información de eventos culturales**
disponible en la API de **Kulturklik** (Open Data Euskadi).

**Fuente:** `https://api.euskadi.eus/culture/events/v1.0` · Licencia de datos **CC BY**
(*Eusko Jaurlaritza / Gobierno Vasco*).

Este notebook es **autocontenido**: incluye el cliente de la API, la recogida de datos,
la limpieza, la ingeniería de variables y el análisis visual. Está diseñado para ser
*schema-agnóstico*: detecta las columnas por su nombre, de modo que sigue funcionando
aunque cambien los campos exactos que devuelve la API.

---
## Índice
0. Configuración e imports
1. Cliente de la API
2. Recogida de **toda** la data
3. Aplanado y primer vistazo
4. Calidad de los datos
5. Limpieza e ingeniería de variables
6. Análisis univariante (categóricas · precio · temporal)
7. Análisis multivariante
8. Distribución geográfica
9. Conclusiones

## 0. Configuración e imports

In [1]:
# Si falta alguna dependencia, descomenta:
# %pip install requests pandas numpy matplotlib seaborn

import logging
import re
import time
import json
from dataclasses import dataclass, field
from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

%matplotlib inline
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (11, 6), "figure.dpi": 110,
                     "axes.titleweight": "bold", "axes.titlesize": 14})
PALETTE = "crest"
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger("kulturklik")
print("Entorno listo · pandas", pd.__version__)

Entorno listo · pandas 2.3.3


## 1. Cliente de la API

Cliente con **paginación** (`_page` + `totalPages`), **reintentos** con backoff y **caché**
en disco. No requiere API key.

In [2]:
BASE_URL = "https://api.euskadi.eus/culture/events/v1.0"

@dataclass
class KulturklikClient:
    base_url: str = BASE_URL
    lang: str | None = "SPANISH"
    timeout: int = 30
    max_retries: int = 4
    backoff_factor: float = 0.8
    sleep_between: float = 0.15
    cache_dir: Path | None = field(default=Path(".cache_kulturklik"))
    _session: requests.Session = field(init=False, repr=False)

    def __post_init__(self):
        self._session = requests.Session()
        retry = Retry(total=self.max_retries, backoff_factor=self.backoff_factor,
                      status_forcelist=(429, 500, 502, 503, 504),
                      allowed_methods=("GET",), raise_on_status=False)
        ad = HTTPAdapter(max_retries=retry)
        self._session.mount("https://", ad); self._session.mount("http://", ad)
        self._session.headers.update({"Accept": "application/json",
                                       "User-Agent": "kulturklik-eda/1.0"})
        if self.cache_dir: self.cache_dir.mkdir(parents=True, exist_ok=True)

    def _get(self, path, params=None):
        url = f"{self.base_url}/{path.lstrip('/')}"
        params = dict(params or {})
        if self.lang: params.setdefault("_lang", self.lang)
        r = self._session.get(url, params=params, timeout=self.timeout)
        if r.status_code == 404: return None
        r.raise_for_status()
        if not r.content: return None
        try: return r.json()
        except json.JSONDecodeError: return None

    @staticmethod
    def _items(payload):
        if payload is None: return [], None
        if isinstance(payload, list): return payload, None
        if isinstance(payload, dict):
            tp = payload.get("totalPages")
            for v in payload.values():
                if isinstance(v, list) and (not v or isinstance(v[0], dict)):
                    return v, tp
            return [payload], tp
        return [], None

    def _paginated(self, path, params=None, max_pages=None):
        params = dict(params or {}); page = 1; out = []; total = None
        while True:
            params["_page"] = page
            items, tp = self._items(self._get(path, params))
            if tp is not None: total = tp
            out.extend(items)
            if not items: break
            if total is not None and page >= total: break
            if total is None: break
            if max_pages and page >= max_pages: break
            page += 1; time.sleep(self.sleep_between)
        return out

    # --- catalogos ---
    def get_event_types(self):   return self._paginated("eventType")
    def get_provinces(self):     return self._paginated("provinces")
    def get_municipalities(self):return self._paginated("municipalities")
    # --- eventos ---
    def get_upcoming_events(self): return self._paginated("events/upcoming")
    def get_events_by_date(self, year, month=None, day=None):
        parts = ["events", "byDate", str(year)]
        if month is not None: parts.append(str(month))
        if day is not None:   parts.append(str(day))
        return self._paginated("/".join(parts))

    def fetch_events_range(self, start, end, granularity="day", use_cache=True):
        key = f"events_{start:%Y%m%d}_{end:%Y%m%d}_{granularity}"
        cf = (self.cache_dir / f"{key}.json") if (self.cache_dir and use_cache) else None
        if cf and cf.exists():
            logger.info("cache: %s", cf); return json.loads(cf.read_text("utf-8"))
        recs = []
        for d in self._periods(start, end, granularity):
            try:
                b = (self.get_events_by_date(d.year, d.month) if granularity == "month"
                     else self.get_events_by_date(d.year, d.month, d.day))
            except requests.RequestException as e:
                logger.warning("%s: %s", d, e); b = []
            if b: recs.extend(b)
            time.sleep(self.sleep_between)
        if cf: cf.write_text(json.dumps(recs, ensure_ascii=False), "utf-8")
        return recs

    @staticmethod
    def _periods(start, end, gran):
        if gran == "day":
            cur = start
            while cur <= end: yield cur; cur += timedelta(days=1)
        else:
            y, m = start.year, start.month
            while (y, m) <= (end.year, end.month):
                yield date(y, m, 1)
                m = 1 if m == 12 else m + 1
                y = y + 1 if m == 1 else y

client = KulturklikClient()
print("Cliente instanciado.")

Cliente instanciado.


## 2. Recogida de **toda** la data

Descargamos:
- **Catálogos**: tipos de evento, provincias y municipios.
- **Eventos**: un rango amplio de fechas + los eventos próximos, combinados y deduplicados.

Ajusta `START` y `END` para ampliar/reducir la ventana. Por defecto recogemos desde el
**1 de enero del año en curso** hasta **hoy + 120 días**. La caché evita repetir descargas.

> ⚠️ Una ventana amplia con granularidad diaria hace muchas peticiones. Si quieres ir
> más rápido para una primera prueba, usa `granularity="month"` o reduce el rango.

In [22]:
provinces_all = client.get_provinces()

PROVINCE_IDS = {-2, 1, 48, 20}
provinces = [p for p in provinces_all if p.get("provinceId") in PROVINCE_IDS]

print(f"Provincias seleccionadas ({len(provinces)}):")
for p in provinces:
    print(f"  {p.get('provinceId'):>4} — {p.get('nameEs') or p.get('name') or p}")

Provincias seleccionadas (4):
    -2 — Online
     1 — Araba/Álava
    48 — Bizkaia
    20 — Gipuzkoa


In [24]:
# ---------------- CONFIGURACIÓN DE RECOGIDA ----------------
START = date(date.today().year, 5, 1)        # inicio del año en curso
END   = date.today() + timedelta(days=1)   # +1 día hacia el futuro
GRANULARITY = "day"                           # "day" (fiable) | "month" (rápido)
# -----------------------------------------------------------
# 2.1 Catálogos
event_types   = client.get_event_types()
municipalities= client.get_municipalities()

# Limita a las 3 provincias con más eventos (si la API devuelve event_count)
if provinces and "event_count" in provinces[0]:
    provinces = sorted(provinces, key=lambda x: x.get("event_count", 0), reverse=True)[:3]

print(f"Catálogos -> tipos: {len(event_types)} | provincias: {len(provinces)} | municipios: {len(municipalities)}")

# 2.2 Eventos del rango + próximos
events_range = client.fetch_events_range(START, END, granularity=GRANULARITY)
events_upcoming = client.get_upcoming_events()
print(f"Eventos -> rango: {len(events_range)} | próximos: {len(events_upcoming)}")

# 2.3 Combinar todo
all_events = events_range + events_upcoming
print(f"Total bruto combinado: {len(all_events)} registros")

Catálogos -> tipos: 15 | provincias: 4 | municipios: 336
Eventos -> rango: 15335 | próximos: 2698
Total bruto combinado: 18033 registros


## 3. Aplanado y primer vistazo

In [25]:
# Aplanamos posibles estructuras anidadas (p.ej. municipality.name -> municipality_name)
df = pd.json_normalize(all_events, sep="_")
print("Dimensiones:", df.shape)
print("\nColumnas disponibles:")
print(list(df.columns))
df.head()

Dimensiones: (18033, 43)

Columnas disponibles:
['id', 'type', 'typeEs', 'typeEu', 'nameEs', 'nameEu', 'startDate', 'endDate', 'publicationDate', 'language', 'openingHoursEs', 'openingHoursEu', 'sourceNameEs', 'sourceNameEu', 'sourceUrlEs', 'sourceUrlEu', 'priceEs', 'priceEu', 'purchaseUrlEs', 'purchaseUrlEu', 'descriptionEs', 'descriptionEu', 'municipalityEs', 'municipalityEu', 'municipalityLatitude', 'municipalityLongitude', 'municipalityNoraCode', 'provinceNoraCode', 'establishmentEs', 'establishmentEu', 'urlEventEs', 'urlEventEu', 'urlNameEs', 'urlNameEu', 'images', 'attachment', 'companyEs', 'companyEu', 'placeEs', 'placeEu', 'online', 'urlOnlineEs', 'urlOnlineEu']


,id,type,typeEs,typeEu,nameEs,nameEu,startDate,endDate,publicationDate,language,openingHoursEs,openingHoursEu,sourceNameEs,sourceNameEu,sourceUrlEs,sourceUrlEu,priceEs,priceEu,purchaseUrlEs,purchaseUrlEu,descriptionEs,descriptionEu,municipalityEs,municipalityEu,municipalityLatitude,municipalityLongitude,municipalityNoraCode,provinceNoraCode,establishmentEs,establishmentEu,urlEventEs,urlEventEu,urlNameEs,urlNameEu,images,attachment,companyEs,companyEu,placeEs,placeEu,online,urlOnlineEs,urlOnlineEu
0,2025110709514014,1,Concierto,Kontzertua,JOHN POLLÕN (1 de mayo - Bilbao),JOHN POLLÕN (maiatzak 1 - Bilbo),2026-05-01T00:00:00Z,2026-05-01T00:00:00Z,2026-04-29T10:45:11Z,ES,21:30,21:30,santana27.com,santana27.com,https://www.santana27.com/conciertos/11-evento...,https://www.santana27.com/conciertos/11-evento...,28 €,28 €,https://entradas.codetickets.com/entradas/john...,https://entradas.codetickets.com/entradas/john...,<p><strong>JOHN POLL&Otilde;N</strong>&nbsp;es...,<p><strong>JOHN</strong>&nbsp;<strong>POLL&Oti...,Bilbao,Bilbao,43.256963,-2.923441,167,48,Santana 27 - Fever,Santana 27 - Fever,http://www.santana27.com/,http://www.santana27.com/,Santana 27 - Fever,Santana 27 - Fever,"[{'imageId': '1136994', 'imageFileName': '62.j...",[],NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025121612345298,1,Concierto,Kontzertua,"Orquesta Sinfónica de Navarra: ""Sinfonía de lo...","Nafarroako Orkestra Sinfonikoa: ""Jostailuen Si...",2026-05-01T00:00:00Z,2026-05-01T00:00:00Z,2025-12-16T12:35:17Z,NA,10:30 + 12:00,10:30 + 12:00,baluarte.com,baluarte.com,https://www.baluarte.com/es/agenda/evento/sinf...,https://www.baluarte.com/eu/agenda/evento/sinf...,10€,10€,https://entradasbaluarte.nicdo.es/selection/ev...,https://entradasbaluarte.nicdo.es/selection/ev...,<p>En el marco de su Programa Educativo y Soci...,<p>Baluarte Fundazioaren Hezkuntza eta Gizarte...,Pamplona/Iruña,Pamplona/Iruña,42.817643,-1.648541,476,31,Auditorio Baluarte,Baluarte Auditorioa,http://www.baluarte.com/,http://www.baluarte.com/eus/hasiera,Auditorio Baluarte,Baluarte Auditorioa,"[{'imageId': '1092968', 'imageFileName': 'sinf...",[],NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025122909033742,16,Fiestas,Jaiak,Programa Fiestas San Prudencio y Estíbaliz 202...,San Prudentzioren eta Estibalizko Andra Mariar...,2026-04-24T00:00:00Z,2026-05-01T00:00:00Z,2026-04-26T08:40:02Z,NaN,NaN,NaN,araba.eus/,araba.eus/,https://prentsa.araba.eus/es/-/las-fiestas-de-...,https://prentsa.araba.eus/es/-/las-fiestas-de-...,NaN,NaN,NaN,NaN,<p>Las fiestas de &Aacute;lava 2026 arrancar&a...,<p>Jaiak ofizialki apirilaren 24an hasiko dira...,Vitoria-Gasteiz,Vitoria-Gasteiz,42.846718,-2.671635,46,1,NaN,NaN,NaN,NaN,NaN,NaN,"[{'imageId': '1135712', 'imageFileName': 'Prog...",[],NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025122912451297,2,Teatro,Antzerkia,"""Ezer ez delako amaitzen""","""Ezer ez delako amaitzen""",2026-05-01T00:00:00Z,2026-05-01T00:00:00Z,2026-04-29T13:22:52Z,EU,19:00,19:00,loraldia.eus/,loraldia.eus/,https://loraldia.eus/es/ezer-ez-delako-amaitze...,https://loraldia.eus/es/ezer-ez-delako-amaitze...,5 €,5 €,https://tickets.kutxabank.es/#/es/detalle/2026...,https://tickets.kutxabank.es/#/es/detalle/2026...,"<p>Tras el golpe fascista de 1936, miles de ni...","<p>1936ko kolpe faxistaren ondoren, milaka hau...",Durango,Durango,43.169068,-2.632655,171,48,Centro Cultural San Agustín,San Agustin Kultur Gunea,http://www.durango-udala.net/portalDurango/p_1...,http://www.durango-udala.net/portalDurango/p_1...,Centro Cultural San Agustín,San Agustin Kultur Gunea,"[{'imageId': '1137260', 'imageFileName': '3242...",[],NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026010913323369,2,Teatro,Antzerkia,"Unarisamás: ""Monólogos All Star""","Unarisamás: ""Monólogos All Star""",2026-05-01T00:00:00Z,2026-05-01T00:00:00Z,2026-01-09T13:32:56Z,ES,20:00,20:00,teatrocampos.com,teatrocampos.com,https://www.teatrocampos.com/espectaculo/monol...,https://www.teatrocampos.com/espectaculo/monol...,18 / 24 €,18 / 24 €,https://entradas.teatrocampos.com/espectaculo/...,https://entr

In [27]:
df["typeEs"].unique()

array(['Concierto', 'Fiestas', 'Teatro', 'Danza', 'Conferencia', 'Otro',
       'Cine y audiovisuales', 'Bertsolarismo', 'Exposición', 'Festival',
       'Feria', 'Formación', 'Concurso', 'Presentación',
       'Eventos/jornadas'], dtype=object)

In [28]:
# seleciion de eventos de tipo Formación
df_formacion = df[df["typeEs"] == "Formación"]
df_formacion


,id,type,typeEs,typeEu,nameEs,nameEu,startDate,endDate,publicationDate,language,openingHoursEs,openingHoursEu,sourceNameEs,sourceNameEu,sourceUrlEs,sourceUrlEu,priceEs,priceEu,purchaseUrlEs,purchaseUrlEu,descriptionEs,descriptionEu,municipalityEs,municipalityEu,municipalityLatitude,municipalityLongitude,municipalityNoraCode,provinceNoraCode,establishmentEs,establishmentEu,urlEventEs,urlEventEu,urlNameEs,urlNameEu,images,attachment,companyEs,companyEu,placeEs,placeEu,online,urlOnlineEs,urlOnlineEu
81,2026010309315982,11,Formación,Formakuntza,"Curso ""La Antigua Grecia: el esplendor del mun...","""Antzinako Grezia: mundu klasikoaren distira"" ...",2026-02-02T00:00:00Z,2026-05-04T00:00:00Z,2026-01-05T09:47:22Z,ES,18:00-19:00 o 19:30-20:30,18:00-19:00 edo 19:30-20:30,losviajesdeaspasia.com,losviajesdeaspasia.com,https://losviajesdeaspasia.com/tienda/grecia-c...,https://losviajesdeaspasia.com/tienda/grecia-c...,90 €,90 €,https://losviajesdeaspasia.com/tienda/grecia-c...,https://losviajesdeaspasia.com/tienda/grecia-c...,<p>Este curso ofrece un viaje profundo al cora...,"<p><span class=""form-control-text textarea"" st...",Donostia / San Sebastián,Donostia / San Sebastián,43.320725,-1.984449,82,20,NaN,NaN,NaN,NaN,NaN,NaN,"[{'imageId': '1101241', 'imageFileName': '3245...",[],NaN,NaN,Salón de actos de COGITI,COGITI Aretoa,NaN,NaN,NaN
85,2026010514543022,11,Formación,Formakuntza,"Curso ""Arte y Arqueología del Antiguo Egipto""","""Antzinako Egiptoko Artea eta Arkeologia"" ikas...",2026-02-03T00:00:00Z,2026-05-05T00:00:00Z,2026-01-07T09:26:29Z,ES,18:00-19:00 (después disponible en diferido),18:00-19:00 (después disponible en diferido),losviajesdeaspasia.com,losviajesdeaspasia.com,https://losviajesdeaspasia.com/tienda/egipto/,https://losviajesdeaspasia.com/tienda/egipto/,85 €,85 €,https://losviajesdeaspasia.com/tienda/egipto/,https://losviajesdeaspasia.com/tienda/egipto/,<p>Desde la organizaci&oacute;n indican:</p>\r...,"<p><span class=""form-control-text textarea"" st...",NaN,NaN,NaN,NaN,NaN,-3,NaN,NaN,NaN,NaN,NaN,NaN,"[{'imageId': '1101578', 'imageFileName': '8.jp...",[],NaN,NaN,NaN,NaN,1,https://losviajesdeaspasia.com/tienda/egipto/,https://losviajesdeaspasia.com/tienda/egipto/
88,2026022512491829,11,Formación,Formakuntza,"""Artea, inguratzen gaituen munduaren isla gisa""","""Artea, inguratzen gaituen munduaren isla gisa""",2026-01-20T00:00:00Z,2026-05-05T00:00:00Z,2026-02-25T12:49:33Z,NaN,NaN,NaN,guggenheim-bilbao.eus,guggenheim-bilbao.eus,https://www.guggenheim-bilbao.eus/actividades/...,https://www.guggenheim-bilbao.eus/eu/jarduerak...,150 €,150 €,NaN,NaN,<p>Curso de iniciaci&oacute;n al arte contempo...,<p>Arte garaikidea ezagutzeko hastapen-ikastar...,Bilbao,Bilbao,43.256963,-2.923441,167,48,Museo Guggenheim Bilbao,Guggenheim Bilbao Museoa,http://www.guggenheim-bilbao.eus/,http://www.guggenheim-bilbao.eus/,Museo Guggenheim Bilbao,Guggenheim Bilbao Museoa,"[{'imageId': '1118351', 'imageFileName': '39.j...",[],NaN,NaN,NaN,NaN,1,https://www.guggenheim-bilbao.eus/actividades/...,https://www.guggenheim-bilbao.eus/eu/jarduerak...
183,2026011910363932,11,Formación,Formakuntza,"""Otros Egiptos: Alejandría y el mundo grecorro...","""BESTE EGIPTO BATZUK: ALEXANDRIA ETA MUNDU GRE...",2026-02-16T00:00:00Z,2026-05-25T00:00:00Z,2026-01-19T10:38:20Z,ES,NaN,NaN,NaN,NaN,NaN,NaN,75 €,75 €,NaN,NaN,<div><span><strong>OTROS EGIPTOS:</strong>&nbs...,<div><em><strong>BESTE EGIPTO BATZUK:&nbsp;ALE...,Irun,Irun,43.338272,-1.789207,101,20,Museo Romano Oiasso,Oiasso Erromatar Museoa,http://www.oiasso.com/es/,http://www.oiasso.com/eu/,Museo Romano Oiasso,Oiasso Erromatar Museoa,"[{'imageId': '1105842', 'imageFileName': '38.j...",[],NaN,NaN,NaN,NaN,NaN,NaN,NaN
184,2026012518583433,11,Formación,Formakuntza,"""Curso de Iniciación a la fotografía - Primave...","""Curso de Iniciación a la fotografía - Primave...",2026-03-02T00:00:00Z,2026-05-25T00:00:00Z,2026-01-26T08:22:36Z,ES,18:00 - 20:00,18:00 - 20:00,denbora.org,denbora.org,https://www.denbora.org/blog/curs

In [ ]:
df.info()

## 4. Calidad de los datos

Revisamos duplicados, valores ausentes y cardinalidad de cada columna antes de limpiar.

In [ ]:
# Deduplicado por id si existe
def find_col(frame, *kw):
    """Devuelve la 1ª columna cuyo nombre contiene alguna palabra clave."""
    for c in frame.columns:
        if any(k in str(c).lower() for k in kw):
            return c
    return None

id_col = find_col(df, "id")
dups = df.duplicated(subset=[id_col]).sum() if id_col else df.duplicated().sum()
print(f"Filas: {len(df)} | duplicadas por '{id_col}': {dups}")

In [ ]:
# Valores ausentes
miss = df.isna().sum().to_frame("n_nulos")
miss["pct"] = (miss["n_nulos"] / len(df) * 100).round(1)
miss = miss[miss["n_nulos"] > 0].sort_values("n_nulos", ascending=False)
display(miss)

if not miss.empty:
    fig, ax = plt.subplots(figsize=(10, max(3, 0.45*len(miss))))
    sns.barplot(x=miss["pct"].values, y=miss.index.astype(str),
                hue=miss.index.astype(str), palette=PALETTE, legend=False, ax=ax)
    ax.set(title="% de valores ausentes por columna", xlabel="% nulos", ylabel="")
    plt.show()

In [ ]:
# Cardinalidad (nº de valores únicos) — ayuda a distinguir categóricas de texto libre
card = df.nunique(dropna=True).sort_values(ascending=False).to_frame("valores_unicos")
display(card)

## 5. Limpieza e ingeniería de variables

- Eliminamos duplicados.
- Convertimos **fechas** y **precio** a tipos correctos.
- Derivamos variables útiles: año, mes, día de la semana, fin de semana, duración,
  gratuito/de pago y longitud de la descripción.

In [ ]:
df = df.drop_duplicates(subset=[id_col] if id_col else None).reset_index(drop=True)

# Detección de columnas clave
c_start = find_col(df, "startdate", "fechainicio", "start", "fecha", "data")
c_end   = find_col(df, "enddate", "fechafin", "end")
c_price = find_col(df, "price", "precio", "prezio")
c_desc  = find_col(df, "description", "descrip", "azalpen")
c_type  = find_col(df, "eventtype", "tipo", "mota", "type", "categor")
c_prov  = find_col(df, "province", "provin", "lurral")
c_muni  = find_col(df, "municipality_name", "municip", "udal", "localidad", "town", "city")
c_estab = find_col(df, "establish", "espacio", "lugar", "venue", "sala")
c_lat   = find_col(df, "latitud", "lat")
c_lon   = find_col(df, "longitud", "lon", "lng")

print("Columnas detectadas:")
for n, c in [("inicio",c_start),("fin",c_end),("precio",c_price),("descripción",c_desc),
             ("tipo",c_type),("provincia",c_prov),("municipio",c_muni),
             ("establecimiento",c_estab),("lat",c_lat),("lon",c_lon)]:
    print(f"  {n:16s}: {c}")

In [ ]:
# Fechas
for c in (c_start, c_end):
    if c: df[c] = pd.to_datetime(df[c], errors="coerce", format="mixed")

# Precio -> euros numéricos + gratuito
if c_price:
    df["price_eur"] = (df[c_price].astype(str)
                       .str.replace(r"[^\d,.\-]", "", regex=True)
                       .str.replace(",", ".", regex=False)
                       .replace("", np.nan)
                       .pipe(pd.to_numeric, errors="coerce"))
    df["is_free"] = df["price_eur"].fillna(0).eq(0)

# Variables temporales
if c_start:
    dn = ["Lunes","Martes","Miércoles","Jueves","Viernes","Sábado","Domingo"]
    df["year"]        = df[c_start].dt.year
    df["month"]       = df[c_start].dt.month
    df["weekday"]     = df[c_start].dt.dayofweek.map(dict(enumerate(dn)))
    df["is_weekend"]  = df[c_start].dt.dayofweek >= 5

# Duración
if c_start and c_end:
    df["duration_days"] = (df[c_end] - df[c_start]).dt.days.clip(lower=0)

# Longitud de descripción
if c_desc:
    df["desc_len"] = df[c_desc].astype(str).str.len()

# Categóricas de baja cardinalidad -> dtype category (memoria + velocidad)
for c in (c_type, c_prov, c_muni, c_estab):
    if c and df[c].nunique() < 0.5 * len(df):
        df[c] = df[c].astype("category")

print("Dataset limpio:", df.shape)
nuevas = [c for c in ["price_eur","is_free","year","month","weekday","is_weekend","duration_days","desc_len"] if c in df.columns]
print("Variables derivadas:", nuevas)
df[[c for c in (c_start, c_type, c_prov, c_muni, "price_eur") if c]].head()

## 6. Análisis univariante

### 6.1 Variables categóricas

In [ ]:
cat_cols = [c for c in (c_type, c_prov, c_muni, c_estab) if c]
weekday_order = ["Lunes","Martes","Miércoles","Jueves","Viernes","Sábado","Domingo"]

for c in cat_cols:
    vc = df[c].astype(str).value_counts().head(15)
    fig, ax = plt.subplots(figsize=(10, max(3.5, 0.45*len(vc))))
    sns.barplot(x=vc.values, y=vc.index, hue=vc.index, palette=PALETTE, legend=False, ax=ax)
    ax.set(title=f"Top {len(vc)} — {c}", xlabel="nº de eventos", ylabel="")
    for i, v in enumerate(vc.values):
        ax.text(v, i, f" {v}", va="center", fontsize=9)
    plt.show()
    print(f"{c}: {df[c].nunique()} valores únicos · más frecuente: {vc.index[0]} ({vc.iloc[0]})\n")

### 6.2 Precio

In [ ]:
if "price_eur" in df.columns and df["price_eur"].notna().any():
    print(df["price_eur"].describe().round(2))
    free_pct = df["is_free"].mean()*100 if "is_free" in df.columns else np.nan
    print(f"\nEventos gratuitos: {free_pct:.1f}%")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(df["price_eur"].dropna(), kde=True, color=sns.color_palette(PALETTE)[2], ax=axes[0])
    axes[0].set(title="Distribución del precio (€)", xlabel="precio")
    # solo precios > 0 para ver la cola sin el pico de gratuitos
    de_pago = df.loc[df["price_eur"] > 0, "price_eur"]
    sns.boxplot(x=de_pago, color=sns.color_palette(PALETTE)[3], ax=axes[1])
    axes[1].set(title="Precio de eventos de pago (€)", xlabel="precio")
    plt.tight_layout(); plt.show()
else:
    print("No hay columna de precio numérica utilizable.")

### 6.3 Distribución temporal

In [ ]:
if c_start and df[c_start].notna().any():
    s = df[c_start].dropna()
    print(f"Rango temporal: {s.min():%Y-%m-%d} → {s.max():%Y-%m-%d}")

    # Serie semanal
    fig, ax = plt.subplots(figsize=(12, 5))
    df.set_index(c_start).resample("W").size().plot(ax=ax, lw=2, color=sns.color_palette(PALETTE)[3])
    ax.set(title="Eventos por semana", xlabel="fecha", ylabel="nº de eventos")
    plt.show()

    # Por mes y por día de la semana
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    by_month = df["month"].value_counts().sort_index()
    sns.barplot(x=by_month.index, y=by_month.values, hue=by_month.index, palette=PALETTE, legend=False, ax=axes[0])
    axes[0].set(title="Eventos por mes", xlabel="mes", ylabel="nº de eventos")
    by_dow = df["weekday"].value_counts().reindex(weekday_order).fillna(0)
    sns.barplot(x=by_dow.index, y=by_dow.values, hue=by_dow.index, palette=PALETTE, legend=False, ax=axes[1])
    axes[1].set(title="Eventos por día de la semana", xlabel="", ylabel="")
    axes[1].tick_params(axis="x", rotation=40)
    plt.tight_layout(); plt.show()
else:
    print("No hay columna de fecha utilizable.")

## 7. Análisis multivariante

Relaciones entre variables: dónde se concentra cada tipo de evento, cómo varía el precio
por tipo y cuándo (mes × día de la semana) hay más actividad.

In [ ]:
# 7.1 Tipo de evento × provincia
if c_type and c_prov:
    ct = pd.crosstab(df[c_type], df[c_prov])
    fig, ax = plt.subplots(figsize=(min(2+1.4*ct.shape[1], 14), max(4, 0.5*ct.shape[0])))
    sns.heatmap(ct, annot=True, fmt="d", cmap="crest", ax=ax)
    ax.set(title="Tipo de evento × Provincia", xlabel="", ylabel="")
    plt.show()

In [ ]:
# 7.2 Precio por tipo de evento
if c_type and "price_eur" in df.columns and df["price_eur"].notna().any():
    order = df.groupby(c_type, observed=True)["price_eur"].median().sort_values().index
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.boxplot(data=df[df["price_eur"] > 0], x=c_type, y="price_eur",
                order=order, hue=c_type, palette=PALETTE, legend=False, ax=ax)
    ax.set(title="Precio (€) por tipo de evento — solo de pago", xlabel="", ylabel="precio")
    ax.tick_params(axis="x", rotation=40)
    plt.show()

In [ ]:
# 7.3 Mapa de calor mes × día de la semana
if c_start and df[c_start].notna().any():
    piv = df.pivot_table(index="weekday", columns="month",
                         values=(id_col or df.columns[0]), aggfunc="count")
    piv = piv.reindex(weekday_order)
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.heatmap(piv, cmap="crest", linewidths=.5, ax=ax)
    ax.set(title="Intensidad de eventos: día de la semana × mes", xlabel="mes", ylabel="")
    plt.show()

## 8. Distribución geográfica

Si los eventos incluyen coordenadas, las representamos en un scatter. Para un mapa
interactivo real sobre callejero puedes usar `folium` (`pip install folium`).

In [ ]:
if c_lat and c_lon and df[c_lat].notna().any():
    geo = df.dropna(subset=[c_lat, c_lon]).copy()
    geo[c_lat] = pd.to_numeric(geo[c_lat], errors="coerce")
    geo[c_lon] = pd.to_numeric(geo[c_lon], errors="coerce")
    geo = geo.dropna(subset=[c_lat, c_lon])
    fig, ax = plt.subplots(figsize=(10, 8))
    sc = ax.scatter(geo[c_lon], geo[c_lat], s=14, alpha=0.35,
                    c=sns.color_palette(PALETTE)[3])
    ax.set(title=f"Localización de {len(geo)} eventos con coordenadas",
           xlabel="longitud", ylabel="latitud")
    ax.set_aspect("equal", adjustable="datalim")
    plt.show()

    # --- Mapa interactivo opcional ---
    # import folium
    # m = folium.Map(location=[geo[c_lat].mean(), geo[c_lon].mean()], zoom_start=9)
    # for _, r in geo.iterrows():
    #     folium.CircleMarker([r[c_lat], r[c_lon]], radius=3).add_to(m)
    # m
else:
    print("No hay coordenadas geográficas en los datos.")

## 9. Conclusiones

Resumen automático de los hallazgos principales. Edítalo con tus observaciones
cualitativas tras revisar los gráficos.

In [ ]:
print("="*60)
print("RESUMEN DEL EDA — KULTURKLIK")
print("="*60)
print(f"Eventos analizados      : {len(df):,}")
if c_start and df[c_start].notna().any():
    print(f"Rango temporal          : {df[c_start].min():%Y-%m-%d} a {df[c_start].max():%Y-%m-%d}")
if c_type:
    top_t = df[c_type].astype(str).value_counts()
    print(f"Tipos de evento         : {df[c_type].nunique()} (top: {top_t.index[0]}, {top_t.iloc[0]})")
if c_muni:
    top_m = df[c_muni].astype(str).value_counts()
    print(f"Municipios              : {df[c_muni].nunique()} (top: {top_m.index[0]}, {top_m.iloc[0]})")
if "is_free" in df.columns:
    print(f"Eventos gratuitos       : {df['is_free'].mean()*100:.1f}%")
if "price_eur" in df.columns and df['price_eur'].notna().any():
    print(f"Precio medio (de pago)  : {df.loc[df['price_eur']>0,'price_eur'].mean():.2f} €")
if "is_weekend" in df.columns:
    print(f"Eventos en fin de semana: {df['is_weekend'].mean()*100:.1f}%")
print("="*60)

# Guardar el dataset limpio
df.to_csv("kulturklik_eventos_limpio.csv", index=False)
print("Dataset limpio guardado en: kulturklik_eventos_limpio.csv")

---
*Notebook generado para el análisis de datos abiertos de Kulturklik (Open Data Euskadi).
Datos bajo licencia CC BY — Eusko Jaurlaritza / Gobierno Vasco.*